# 3. Feature Engineering - 데이팅 데이터 다루기

피처 엔지니어링(Feature Engineering)은 기존 데이터에서 **모델에 유용한 새로운 변수를 만들어내는** 과정이다.
단순히 데이터를 정제하는 것을 넘어, 데이터가 가진 **숨겨진 정보를 끌어내는** 작업이다.

**다루는 내용:**
- 컬럼 이름 일괄 변경 (startswith, endswith, replace)
- 결측치 전략 (종속변수 vs 독립변수)
- 이상치 처리 및 값 정규화 (합계 100 맞추기)
- 파생 변수 생성 (나이차, 인종 일치 점수)
- Rating 계산 (중요도 × 점수)
- 조화 평균을 활용한 최종 점수 산출
- 원-핫 인코딩 및 최종 데이터 정리

## 3-1. 데이터 로드 및 탐색

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
dating_df = pd.read_csv('/Users/chankyulee/Desktop/ModuLABS/02_DataPreprocessing/Data/csv/dating.csv')

In [3]:
# 컬럼이 많으므로 최대 출력 개수를 50으로 설정
pd.set_option('display.max_columns', 50)

In [4]:
dating_df.head(10)

,gender,age,age_o,race,race_o,importance_same_race,importance_same_religion,pref_o_attractive,pref_o_sincere,pref_o_intelligence,pref_o_funny,pref_o_ambitious,pref_o_shared_interests,attractive_o,sincere_o,intelligence_o,funny_o,ambitous_o,shared_interests_o,attractive_important,sincere_important,intellicence_important,funny_important,ambtition_important,shared_interests_important,attractive_partner,sincere_partner,intelligence_partner,funny_partner,ambition_partner,shared_interests_partner,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match
0,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,35.00,20.00,20.00,20.00,0.00,5.00,6.0,8.0,8.0,8.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,6.0,9.0,7.0,7.0,6.0,5.0,0.14,3.0,2.0,7.0,6.0,0
1,female,21.0,22.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,60.00,0.00,0.00,40.00,0.00,0.00,7.0,8.0,10.0,7.0,7.0,5.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,8.0,7.0,8.0,5.0,6.0,0.54,3.0,2.0,7.0,5.0,0
2,female,21.0,22.0,Asian/PacificIslander/Asian-American,Asian/PacificIslander/Asian-American,2.0,4.0,19.00,18.00,19.00,18.00,14.00,12.00,10.0,10.0,10.0,10.0,10.0,10.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,8.0,9.0,8.0,5.0,7.0,0.16,3.0,2.0,7.0,NaN,1
3,female,21.0,23.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,30.00,5.00,15.00,40.00,5.00,5.00,7.0,8.0,9.0,8.0,9.0,8.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,6.0,8.0,7.0,6.0,8.0,0.61,3.0,2.0,7.0,6.0,1
4,female,21.0,24.0,Asian/PacificIslander/Asian-American,Latino/HispanicAmerican,2.0,4.0,30.00,10.00,20.00,10.00,10.00,20.00,8.0,7.0,9.0,6.0,9.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,6.0,7.0,7.0,6.0,6.0,0.21,3.0,2.0,6.0,6.0,1
5,female,21.0,25.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,50.00,0.00,30.00,10.00,0.00,10.00,7.0,7.0,8.0,8.0,7.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,4.0,9.0,7.0,4.0,6.0,4.0,0.25,3.0,2.0,6.0,5.0,0
6,female,21.0,30.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,35.00,15.00,25.00,10.00,5.00,10.00,3.0,6.0,7.0,5.0,8.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,6.0,7.0,4.0,6.0,7.0,0.34,3.0,2.0,6.0,5.0,0
7,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,33.33,11.11,11.11,11.11,11.11,22.22,6.0,7.0,5.0,6.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,4.0,9.0,7.0,6.0,5.0,6.0,0.50,3.0,2.0,6.0,7.0,0
8,female,21.0,28.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,50.00,0.00,25.00,10.00,0.00,15.00,7.0,7.0,8.0,8.0,8.0,9.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,6.0,8.0,9.0,8.0,8.0,0.28,3.0,2.0,7.0,7.0,1
9,female,21.0,24.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,100.00,0.00,0.00,0.00,0.00,0.00,6.0,6.0,6.0,6.0,6.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,6.0,6.0,8.0,10.0,8.0,-0.36,3.0,2.0,6.0,6.0,0


In [5]:
dating_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8378 entries, 0 to 8377
Data columns (total 37 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   gender                         8378 non-null   object 
 1   age                            8283 non-null   float64
 2   age_o                          8274 non-null   float64
 3   race                           8315 non-null   object 
 4   race_o                         8305 non-null   object 
 5   importance_same_race           8299 non-null   float64
 6   importance_same_religion       8299 non-null   float64
 7   pref_o_attractive              8289 non-null   float64
 8   pref_o_sincere                 8289 non-null   float64
 9   pref_o_intelligence            8289 non-null   float64
 10  pref_o_funny                   8280 non-null   float64
 11  pref_o_ambitious               8271 non-null   float64
 12  pref_o_shared_interests        8249 non-null   f

In [6]:
dating_df.describe()

,age,age_o,importance_same_race,importance_same_religion,pref_o_attractive,pref_o_sincere,pref_o_intelligence,pref_o_funny,pref_o_ambitious,pref_o_shared_interests,attractive_o,sincere_o,intelligence_o,funny_o,ambitous_o,shared_interests_o,attractive_important,sincere_important,intellicence_important,funny_important,ambtition_important,shared_interests_important,attractive_partner,sincere_partner,intelligence_partner,funny_partner,ambition_partner,shared_interests_partner,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match
count,8283.000000,8274.000000,8299.000000,8299.000000,8289.000000,8289.000000,8289.000000,8280.000000,8271.000000,8249.000000,8166.000000,8091.000000,8072.000000,8018.000000,7656.000000,7302.000000,8299.000000,8299.000000,8299.000000,8289.000000,8279.000000,8257.000000,8176.000000,8101.000000,8082.000000,8028.000000,7666.000000,7311.000000,8220.000000,8277.000000,1800.000000,8138.000000,8069.000000,8378.000000
mean,26.358928,26.364999,3.784793,3.651645,22.495347,17.396867,20.270759,17.459714,10.685375,11.845930,6.190411,7.175256,7.369301,6.400599,6.778409,5.474870,22.514632,17.396389,20.265613,17.457043,10.682539,11.845111,6.189995,7.175164,7.368597,6.400598,6.777524,5.474559,0.196010,5.534131,5.570556,6.134087,5.207523,0.164717
std,3.566763,3.563648,2.845708,2.805237,12.569802,7.044003,6.782895,6.085526,6.126544,6.362746,1.950305,1.740575,1.550501,1.954078,1.794080,2.156163,12.587674,7.046700,6.783003,6.085239,6.124888,6.362154,1.950169,1.740315,1.550453,1.953702,1.794055,2.156363,0.303539,1.734059,4.762569,1.841285,2.129565,0.370947
min,18.000000,18.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.830000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,24.000000,24.000000,1.000000,1.000000,15.000000,15.000000,17.390000,15.000000,5.000000,9.520000,5.000000,6.000000,6.000000,5.000000,6.000000,4.000000,15.000000,15.000000,17.390000,15.000000,5.000000,9.520000,5.000000,6.000000,6.000000,5.000000,6.000000,4.000000,-0.020000,5.000000,2.000000,5.000000,4.000000,0.000000
50%,26.000000,26.000000,3.000000,3.000000,20.000000,18.370000,20.000000,18.000000,10.000000,10.640000,6.000000,7.000000,7.000000,7.000000,7.000000,6.000000,20.000000,18.180000,20.000000,18.000000,10.000000,10.640000,6.000000,7.000000,7.000000,7.000000,7.000000,6.000000,0.210000,6.000000,4.000000,6.000000,5.000000,0.000000
75%,28.000000,28.000000,6.000000,6.000000,25.000000,20.000000,23.810000,20.000000,15.000000,16.000000,8.000000,8.000000,8.000000,8.000000,8.000000,7.000000,25.000000,20.000000,23.810000,20.000000,15.000000,16.000000,8.000000,8.000000,8.000000,8.000000,8.000000,7.000000,0.430000,7.000000,8.000000,7.000000,7.000000,0.000000
max,55.000000,55.000000,10.000000,10.000000,100.000000,60.000000,50.000000,50.000000,53.000000,30.000000,10.500000,10.000000,10.000000,11.000000,10.000000,10.000000,100.000000,60.000000,50.000000,50.000000,53.000000,30.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,0.910000,10.000000,20.000000,10.000000,10.000000,1.000000


## 3-2. 컬럼 이름 일괄 변경

원본 데이터의 컬럼명이 규칙적이지 않아서, 분석하기 편하게 접두사를 통일한다.

**변경 규칙:**

| 원본 패턴 | 변경 후 | 의미 |
|-----------|---------|------|
| `pref_o_xxx` | `o_important_xxx` | 상대방이 생각하는 중요도 |
| `xxx_o` | `o_score_xxx` | 상대방에 대한 평가 점수 |
| `xxx_important` | `i_important_xxx` | 내가 생각하는 중요도 |
| `xxx_partner` | `i_score_xxx` | 나에 대한 평가 점수 |

**사용하는 문자열 메서드:**
- `.startswith('문자열')` : 해당 문자로 시작하는지 확인 (True/False)
- `.endswith('문자열')` : 해당 문자로 끝나는지 확인
- `.replace('기존', '신규')` : 문자열 치환

In [7]:
dating_df.columns

Index(['gender', 'age', 'age_o', 'race', 'race_o', 'importance_same_race',
       'importance_same_religion', 'pref_o_attractive', 'pref_o_sincere',
       'pref_o_intelligence', 'pref_o_funny', 'pref_o_ambitious',
       'pref_o_shared_interests', 'attractive_o', 'sincere_o',
       'intelligence_o', 'funny_o', 'ambitous_o', 'shared_interests_o',
       'attractive_important', 'sincere_important', 'intellicence_important',
       'funny_important', 'ambtition_important', 'shared_interests_important',
       'attractive_partner', 'sincere_partner', 'intelligence_partner',
       'funny_partner', 'ambition_partner', 'shared_interests_partner',
       'interests_correlate', 'expected_happy_with_sd_people',
       'expected_num_interested_in_me', 'like', 'guess_prob_liked', 'match'],
      dtype='object')

In [8]:
'string'.startswith('b')

False

In [9]:
'string'.endswith('g')

True

In [10]:
'string'.replace('s', 'a')

'atring'

In [11]:
# 컬럼 이름 변경 로직
# for문으로 모든 컬럼을 순회하면서 규칙에 따라 접두사를 변경
new_cols = []

for i in dating_df.columns:
    if i.startswith('pref_o'):
        i = i.replace('pref_o', 'o_important')
    elif i.endswith('_o'):
        i = 'o_score_' + i.replace('_o', '')
    elif i.endswith('_important'):
        i = 'i_important_' + i.replace('_important', '')
    elif i.endswith('_partner'):
        i = 'i_score_' + i.replace('_partner', '')
    new_cols.append(i)

In [12]:
dating_df.columns = new_cols

In [13]:
dating_df.head()

,gender,age,o_score_age,race,o_score_race,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,i_score_attractive,i_score_sincere,i_score_intelligence,i_score_funny,i_score_ambition,i_score_shared_interests,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match
0,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,35.0,20.0,20.0,20.0,0.0,5.0,6.0,8.0,8.0,8.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,6.0,9.0,7.0,7.0,6.0,5.0,0.14,3.0,2.0,7.0,6.0,0
1,female,21.0,22.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,60.0,0.0,0.0,40.0,0.0,0.0,7.0,8.0,10.0,7.0,7.0,5.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,8.0,7.0,8.0,5.0,6.0,0.54,3.0,2.0,7.0,5.0,0
2,female,21.0,22.0,Asian/PacificIslander/Asian-American,Asian/PacificIslander/Asian-American,2.0,4.0,19.0,18.0,19.0,18.0,14.0,12.0,10.0,10.0,10.0,10.0,10.0,10.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,8.0,9.0,8.0,5.0,7.0,0.16,3.0,2.0,7.0,NaN,1
3,female,21.0,23.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,30.0,5.0,15.0,40.0,5.0,5.0,7.0,8.0,9.0,8.0,9.0,8.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,6.0,8.0,7.0,6.0,8.0,0.61,3.0,2.0,7.0,6.0,1
4,female,21.0,24.0,Asian/PacificIslander/Asian-American,Latino/HispanicAmerican,2.0,4.0,30.0,10.0,20.0,10.0,10.0,20.0,8.0,7.0,9.0,6.0,9.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,6.0,7.0,7.0,6.0,6.0,0.21,3.0,2.0,6.0,6.0,1


- `race_o`와 `age_o`는 상대방 정보이므로 `o_score_` 접두사가 아닌 원래 의미를 유지

In [14]:
dating_df = dating_df.rename({'o_score_race': 'race_o', 'o_score_age': 'age_o'}, axis = 1)

## 3-3. 결측치 처리

결측치 처리 전략은 **종속변수와 독립변수에 따라 다르다:**

- **종속변수** (예: `match`): 예측해야 할 **정답지**이므로 결측치가 있으면 반드시 **삭제**. 평균이나 임의값으로 채우면 잘못된 예측으로 이어진다.
- **독립변수**: 분석 목적에 따라 삭제 또는 채우기 가능

여기서는:
1. `o_important`, `i_important` 관련 열에 결측치가 있으면 해당 행 삭제 (중요도 정보가 없으면 분석 불가)
2. 나머지 결측치는 `-99`로 채우기 (응답하지 않은 항목)

In [15]:
dating_df.isna().mean()

gender                           0.000000
age                              0.011339
age_o                            0.012413
race                             0.007520
race_o                           0.008713
importance_same_race             0.009429
importance_same_religion         0.009429
o_important_attractive           0.010623
o_important_sincere              0.010623
o_important_intelligence         0.010623
o_important_funny                0.011697
o_important_ambitious            0.012772
o_important_shared_interests     0.015397
o_score_attractive               0.025304
o_score_sincere                  0.034256
o_score_intelligence             0.036524
o_score_funny                    0.042970
o_score_ambitous                 0.086178
o_score_shared_interests         0.128432
i_important_attractive           0.009429
i_important_sincere              0.009429
i_important_intellicence         0.009429
i_important_funny                0.010623
i_important_ambtition            0

In [16]:
dating_df[dating_df['o_important_attractive'].isna()]

,gender,age,age_o,race,race_o,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,i_score_attractive,i_score_sincere,i_score_intelligence,i_score_funny,i_score_ambition,i_score_shared_interests,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match
511,male,25.0,26.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,9.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0,6.0,6.0,6.0,5.0,2.0,25.0,20.0,25.0,20.0,10.0,0.0,8.0,8.0,7.0,8.0,7.0,3.0,NaN,3.0,2.0,6.0,2.0,0
530,male,30.0,26.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,6.0,5.0,4.0,5.0,2.0,30.0,20.0,10.0,30.0,0.0,10.0,8.0,8.0,8.0,8.0,8.0,8.0,NaN,7.0,3.0,5.0,NaN,0
549,male,23.0,26.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,1.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0,5.0,8.0,5.0,8.0,3.0,20.0,25.0,20.0,15.0,10.0,10.0,7.0,10.0,9.0,8.0,9.0,NaN,NaN,6.0,3.0,8.0,6.0,0
568,male,24.0,26.0,European/Caucasian-American,European/Caucasian-American,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,4.0,6.0,5.0,4.0,4.0,3.0,25.0,35.0,30.0,10.0,0.0,0.0,5.0,6.0,6.0,4.0,7.0,6.0,NaN,3.0,1.0,5.0,2.0,0
587,male,24.0,26.0,European/Caucasian-American,European/Caucasian-American,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,4.0,6.0,7.0,4.0,5.0,2.0,25.0,15.0,25.0,25.0,15.0,15.0,5.0,5.0,5.0,5.0,6.0,5.0,NaN,7.0,5.0,5.0,5.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5649,male,30.0,NaN,Asian/PacificIslander/Asian-American,NaN,6.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,6.0,7.0,8.0,6.0,7.0,4.0,50.0,10.0,10.0,10.0,5.0,15.0,5.0,5.0,6.0,5.0,5.0,5.0,NaN,7.0,NaN,6.0,6.0,0
5669,male,27.0,NaN,European/Caucasian-American,NaN,3.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,8.0,8.0,9.0,7.0,7.0,8.0,30.0,20.0,15.0,15.0,8.0,12.0,5.0,6.0,7.0,6.0,7.0,7.0,NaN,5.0,NaN,6.0,6.0,0
5689,male,25.0,NaN,Asian/PacificIslander/Asian-American,NaN,10.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,10.0,15.0,20.0,15.0,10.0,8.0,9.0,7.0,7.0,7.0,8.0,NaN,5.0,NaN,7.0,6.0,0
5709,male,26.0,NaN,Asian/PacificIslander/Asian-American,NaN,3.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,6.0,6.0,5.0,4.0,4.0,3.0,25.0,20.0,10.0,20.0,15.0,10.0,9.0,9.0,9.0,9.0,7.0,7.0,NaN,6.0,NaN,8.0,7.0,0


In [17]:
# 'o_important' 또는 'i_important'로 시작하는 모든 열의 이름을 drop_cols에 저장
drop_cols = []

for i in dating_df.columns:
    if i.startswith('o_important'):
        drop_cols.append(i)
    elif i.startswith('i_important'):
        drop_cols.append(i)

In [18]:
drop_cols

['o_important_attractive',
 'o_important_sincere',
 'o_important_intelligence',
 'o_important_funny',
 'o_important_ambitious',
 'o_important_shared_interests',
 'i_important_attractive',
 'i_important_sincere',
 'i_important_intellicence',
 'i_important_funny',
 'i_important_ambtition',
 'i_important_shared_interests']

In [19]:
# drop_cols에 지정된 열 중 하나라도 결측치를 포함하는 행을 전부 제거
dating_df = dating_df.dropna(subset = drop_cols)

In [20]:
dating_df.isna().mean()

gender                           0.000000
age                              0.002706
age_o                            0.002706
race                             0.000000
race_o                           0.000000
importance_same_race             0.000000
importance_same_religion         0.000000
o_important_attractive           0.000000
o_important_sincere              0.000000
o_important_intelligence         0.000000
o_important_funny                0.000000
o_important_ambitious            0.000000
o_important_shared_interests     0.000000
o_score_attractive               0.022755
o_score_sincere                  0.031488
o_score_intelligence             0.034071
o_score_funny                    0.040590
o_score_ambitous                 0.084625
o_score_shared_interests         0.127798
i_important_attractive           0.000000
i_important_sincere              0.000000
i_important_intellicence         0.000000
i_important_funny                0.000000
i_important_ambtition            0

In [21]:
# 나머지 결측치는 '응답하지 않음'으로 간주하고 -99로 채우기
dating_df = dating_df.fillna(-99)

## 3-4. 이상치 처리

- 점수 컬럼(score)의 범위는 0~10이어야 하는데, 10을 초과하는 값이 존재
- 10 초과 값을 10으로 클리핑

In [22]:
dating_df.describe()

,age,age_o,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,i_score_attractive,i_score_sincere,i_score_intelligence,i_score_funny,i_score_ambition,i_score_shared_interests,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match
count,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000,8130.000000
mean,26.034932,26.034932,3.776753,3.658303,22.320451,17.433629,20.266113,17.446224,10.705183,11.872082,3.803555,3.841882,3.757380,2.128106,-2.159533,-7.869065,22.320451,17.433629,20.266113,17.446224,10.705183,11.872082,3.803493,3.841882,3.757380,2.127983,-2.159533,-7.869065,0.198866,5.244649,-76.879090,3.270812,1.537761,0.164822
std,7.418562,7.418562,2.844297,2.807314,12.398470,7.017378,6.770061,6.068068,6.101937,6.351582,15.804941,18.622449,19.358834,20.888796,29.495408,34.942787,12.398470,7.017378,6.770061,6.068068,6.101937,6.351582,15.804916,18.622449,19.358834,20.888746,29.495408,34.942787,0.302338,5.697854,42.761973,17.230563,19.345369,0.371042
min,-99.000000,-99.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-99.000000,-99.000000,-99.000000,-99.000000,-99.000000,-99.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-99.000000,-99.000000,-99.000000,-99.000000,-99.000000,-99.000000,-0.830000,-99.000000,-99.000000,-99.000000,-99.000000,0.000000
25%,24.000000,24.000000,1.000000,1.000000,15.000000,15.000000,17.390000,15.000000,5.000000,9.520000,5.000000,6.000000,6.000000,5.000000,5.000000,3.000000,15.000000,15.000000,17.390000,15.000000,5.000000,9.520000,5.000000,6.000000,6.000000,5.000000,5.000000,3.000000,-0.010000,5.000000,-99.000000,5.000000,4.000000,0.000000
50%,26.000000,26.000000,3.000000,3.000000,20.000000,18.370000,20.000000,18.000000,10.000000,10.870000,6.000000,7.000000,7.000000,6.000000,7.000000,5.000000,20.000000,18.370000,20.000000,18.000000,10.000000,10.870000,6.000000,7.000000,7.000000,6.000000,7.000000,5.000000,0.220000,6.000000,-99.000000,6.000000,5.000000,0.000000
75%,28.000000,28.000000,6.000000,6.000000,25.000000,20.000000,23.260000,20.000000,15.000000,16.000000,8.000000,8.000000,8.000000,8.000000,8.000000,7.000000,25.000000,20.000000,23.260000,20.000000,15.000000,16.000000,8.000000,8.000000,8.000000,8.000000,8.000000,7.000000,0.430000,7.000000,-99.000000,7.000000,7.000000,0.000000
max,55.000000,55.000000,10.000000,10.000000,100.000000,60.000000,50.000000,50.000000,53.000000,30.000000,10.500000,10.000000,10.000000,11.000000,10.000000,10.000000,100.000000,60.000000,50.000000,50.000000,53.000000,30.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,0.910000,10.000000,20.000000,10.000000,10.000000,1.000000


In [23]:
dating_df['o_score_attractive'].sort_values()

5499   -99.0
5419   -99.0
8144   -99.0
7499   -99.0
7497   -99.0
        ... 
4898    10.0
4894    10.0
1102    10.0
869     10.0
6216    10.5
Name: o_score_attractive, Length: 8130, dtype: float64

In [24]:
dating_df['o_score_funny'].sort_values()

8235   -99.0
2110   -99.0
8059   -99.0
7508   -99.0
5643   -99.0
        ... 
516     10.0
1495    10.0
478     10.0
4938    10.0
6608    11.0
Name: o_score_funny, Length: 8130, dtype: float64

In [25]:
dating_df['o_score_attractive'] = dating_df['o_score_attractive'].apply(lambda x: 10  if x > 10  else x )

In [26]:
dating_df['o_score_attractive'].max()

np.float64(10.0)

In [27]:
dating_df['o_score_funny'] = dating_df['o_score_funny'].apply(lambda x: 10  if x > 10  else x )

In [28]:
dating_df['o_score_funny'].max()

np.float64(10.0)

## 3-5. 중요도(importance) 합계 정규화

중요도 항목들의 합이 100이 되어야 하는데, 일부 응답자의 합이 100이 아님.

**해결 방법:** 각 항목을 `(100 / 현재 합계) × 항목값`으로 보정
- 예: 합이 95인 경우 → `(100/95) × 각 값` → 비율은 유지하면서 합을 100으로 맞춤

**구현 과정:**
1. `o_important_` 로 시작하는 컬럼들의 합(o_imp_sum)을 구한다
2. 각 값에 `(100 / 합계)`를 곱해 정규화
3. 정규화 후 합계 컬럼(o_imp_sum, i_imp_sum)은 삭제

In [29]:
o_imp = []

for i in dating_df.columns:
    if i.startswith('o_important'):
        o_imp.append(i)

In [30]:
o_imp

['o_important_attractive',
 'o_important_sincere',
 'o_important_intelligence',
 'o_important_funny',
 'o_important_ambitious',
 'o_important_shared_interests']

In [31]:
dating_df['o_imp_sum'] = dating_df[o_imp].sum(axis = 1)

In [32]:
i_imp = []

for i in dating_df.columns:
    if i.startswith('i_important'):
        i_imp.append(i)

In [33]:
i_imp

['i_important_attractive',
 'i_important_sincere',
 'i_important_intellicence',
 'i_important_funny',
 'i_important_ambtition',
 'i_important_shared_interests']

In [34]:
dating_df['i_imp_sum'] = dating_df[i_imp].sum(axis = 1)

In [35]:
dating_df.head()

,gender,age,age_o,race,race_o,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,i_score_attractive,i_score_sincere,i_score_intelligence,i_score_funny,i_score_ambition,i_score_shared_interests,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match,o_imp_sum,i_imp_sum
0,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,35.0,20.0,20.0,20.0,0.0,5.0,6.0,8.0,8.0,8.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,6.0,9.0,7.0,7.0,6.0,5.0,0.14,3.0,2.0,7.0,6.0,0,100.0,100.0
1,female,21.0,22.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,60.0,0.0,0.0,40.0,0.0,0.0,7.0,8.0,10.0,7.0,7.0,5.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,8.0,7.0,8.0,5.0,6.0,0.54,3.0,2.0,7.0,5.0,0,100.0,100.0
2,female,21.0,22.0,Asian/PacificIslander/Asian-American,Asian/PacificIslander/Asian-American,2.0,4.0,19.0,18.0,19.0,18.0,14.0,12.0,10.0,10.0,10.0,10.0,10.0,10.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,8.0,9.0,8.0,5.0,7.0,0.16,3.0,2.0,7.0,-99.0,1,100.0,100.0
3,female,21.0,23.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,30.0,5.0,15.0,40.0,5.0,5.0,7.0,8.0,9.0,8.0,9.0,8.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,6.0,8.0,7.0,6.0,8.0,0.61,3.0,2.0,7.0,6.0,1,100.0,100.0
4,female,21.0,24.0,Asian/PacificIslander/Asian-American,Latino/HispanicAmerican,2.0,4.0,30.0,10.0,20.0,10.0,10.0,20.0,8.0,7.0,9.0,6.0,9.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,6.0,7.0,7.0,6.0,6.0,0.21,3.0,2.0,6.0,6.0,1,100.0,100.0


In [36]:
# 합이 100이 아닌 행 확인
dating_df[dating_df['o_imp_sum'] != 100]

,gender,age,age_o,race,race_o,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,i_score_attractive,i_score_sincere,i_score_intelligence,i_score_funny,i_score_ambition,i_score_shared_interests,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match,o_imp_sum,i_imp_sum
7,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,33.33,11.11,11.11,11.11,11.11,22.22,6.0,7.0,5.0,6.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,4.0,9.0,7.0,6.0,5.0,6.0,0.50,3.0,2.0,6.0,7.0,0,99.99,100.0
17,female,24.0,27.0,European/Caucasian-American,European/Caucasian-American,2.0,5.0,33.33,11.11,11.11,11.11,11.11,22.22,7.0,7.0,7.0,7.0,7.0,5.0,45.0,5.0,25.0,20.0,0.0,5.0,5.0,8.0,7.0,5.0,9.0,5.0,0.07,4.0,5.0,5.0,6.0,0,99.99,100.0
27,female,25.0,27.0,European/Caucasian-American,European/Caucasian-American,8.0,4.0,33.33,11.11,11.11,11.11,11.11,22.22,4.0,5.0,6.0,4.0,6.0,4.0,35.0,10.0,35.0,10.0,10.0,0.0,7.0,9.0,9.0,8.0,9.0,7.0,0.29,4.0,2.0,8.0,7.0,0,99.99,100.0
37,female,23.0,27.0,European/Caucasian-American,European/Caucasian-American,1.0,1.0,33.33,11.11,11.11,11.11,11.11,22.22,6.0,7.0,8.0,6.0,6.0,5.0,20.0,20.0,20.0,20.0,10.0,10.0,5.0,9.0,9.0,5.0,9.0,7.0,0.15,1.0,2.0,6.0,6.0,0,99.99,100.0
47,female,21.0,27.0,European/Caucasian-American,European/Caucasian-American,8.0,1.0,33.33,11.11,11.11,11.11,11.11,22.22,5.0,5.0,6.0,5.0,5.0,5.0,20.0,5.0,25.0,25.0,10.0,15.0,5.0,5.0,7.0,6.0,7.0,5.0,0.07,7.0,10.0,6.0,4.0,0,99.99,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8343,male,27.0,23.0,Black/AfricanAmerican,European/Caucasian-American,2.0,1.0,20.00,25.00,25.00,30.00,5.00,5.00,7.0,6.0,6.0,5.0,5.0,-99.0,40.0,20.0,20.0,20.0,0.0,0.0,8.0,5.0,5.0,7.0,7.0,5.0,0.53,3.0,-99.0,7.0,3.0,0,110.00,100.0
8351,male,27.0,26.0,Black/AfricanAmerican,Latino/HispanicAmerican,2.0,1.0,10.00,10.00,30.00,20.00,10.00,15.00,10.0,5.0,8.0,6.0,9.0,8.0,40.0,20.0,20.0,20.0,0.0,0.0,7.0,7.0,7.0,7.0,7.0,7.0,0.41,3.0,-99.0,8.0,1.0,0,95.00,100.0
8364,male,25.0,30.0,European/Caucasian-American,Latino/HispanicAmerican,1.0,1.0,15.00,20.00,20.00,20.00,5.00,10.00,6.0,5.0,4.0,3.0,6.0,5.0,70.0,0.0,15.0,15.0,0.0,0.0,6.0,9.0,9.0,9.0,8.0,8.0,0.43,10.0,-99.0,7.0,6.0,0,90.00,100.0
8365,male,25.0,23.0,European/Caucasian-American,European/Caucasian-American,1.0,1.0,20.00,25.00,25.00,30.00,5.00,5.00,8.0,7.0,6.0,7.0,7.0,2.0,70.0,0.0,15.0,15.0,0.0,0.0,5.0,8.0,8.0,7.0,7.0,8.0,0.47,10.0,-99.0,6.0,7.0,0,110.00,100.0


- `(100 / 합계) × 각 항목값` 공식으로 비율을 유지하면서 합을 100으로 보정

In [37]:
dating_df[o_imp] = dating_df.apply(lambda x: (100 / x['o_imp_sum']) * x[o_imp], axis = 1)

In [38]:
# 보정 후 합계 확인 → 100이어야 함
dating_df[o_imp].sum(axis=1).max()

np.float64(100.00000000000003)

In [39]:
dating_df[i_imp] = dating_df.apply(lambda x: (100 / x['i_imp_sum']) * x[i_imp], axis = 1)

In [40]:
dating_df[i_imp].sum(axis=1).min()

np.float64(99.99999999999997)

In [41]:
# 정규화 완료 → 합계 컬럼 삭제
dating_df.drop(['o_imp_sum','i_imp_sum'], axis=1, inplace = True)

In [42]:
dating_df.head()

,gender,age,age_o,race,race_o,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,i_score_attractive,i_score_sincere,i_score_intelligence,i_score_funny,i_score_ambition,i_score_shared_interests,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match
0,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,35.0,20.0,20.0,20.0,0.0,5.0,6.0,8.0,8.0,8.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,6.0,9.0,7.0,7.0,6.0,5.0,0.14,3.0,2.0,7.0,6.0,0
1,female,21.0,22.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,60.0,0.0,0.0,40.0,0.0,0.0,7.0,8.0,10.0,7.0,7.0,5.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,8.0,7.0,8.0,5.0,6.0,0.54,3.0,2.0,7.0,5.0,0
2,female,21.0,22.0,Asian/PacificIslander/Asian-American,Asian/PacificIslander/Asian-American,2.0,4.0,19.0,18.0,19.0,18.0,14.0,12.0,10.0,10.0,10.0,10.0,10.0,10.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,8.0,9.0,8.0,5.0,7.0,0.16,3.0,2.0,7.0,-99.0,1
3,female,21.0,23.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,30.0,5.0,15.0,40.0,5.0,5.0,7.0,8.0,9.0,8.0,9.0,8.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,6.0,8.0,7.0,6.0,8.0,0.61,3.0,2.0,7.0,6.0,1
4,female,21.0,24.0,Asian/PacificIslander/Asian-American,Latino/HispanicAmerican,2.0,4.0,30.0,10.0,20.0,10.0,10.0,20.0,8.0,7.0,9.0,6.0,9.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,6.0,7.0,7.0,6.0,6.0,0.21,3.0,2.0,6.0,6.0,1


## 3-6. 파생 변수: 나이차 (Age Gap)

두 사람의 나이 차이를 계산해서 새로운 변수를 만든다.

**주의사항:**
- 결측치를 `-99`로 채웠기 때문에, 그냥 뺄셈하면 비정상적인 음수가 나옴
- `-99`가 포함된 경우 나이차도 `-99`로 표시
- 성별에 따라 뺄셈 방향을 다르게 처리 (남성 기준: 본인 - 상대)

**추가 파생 변수:**
- `age_gap_dir` : 나이차 방향 (positive/negative/zero)
- `age_gap` : 절대값으로 변환 (방향 무관한 순수 나이차)

In [43]:
def age_func(x):
    if x['age'] == -99:
        return -99
    elif x['age_o'] == -99:
        return -99
    elif x['gender'] == 'female':
        return x['age_o'] - x['age']
    else:
        return x['age'] - x['age_o']

In [44]:
dating_df['age_gap'] = dating_df.apply(age_func, axis = 1)

In [45]:
dating_df['age_gap']

0       6.0
1       1.0
2       1.0
3       2.0
4       3.0
       ... 
8372    1.0
8373   -1.0
8374    1.0
8376    3.0
8377    3.0
Name: age_gap, Length: 8130, dtype: float64

In [46]:
# 나이차 방향을 범주형으로 저장
dating_df['age_gap_dir'] = dating_df['age_gap'].apply(lambda x: 'positive' if x > 0 else 'negative' if x < 0 else 'zero')

In [47]:
# 절대값으로 변환 (방향 정보는 age_gap_dir에 이미 저장됨)
dating_df['age_gap'] = abs(dating_df['age_gap'])

In [48]:
dating_df.head(10)

,gender,age,age_o,race,race_o,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,i_score_attractive,i_score_sincere,i_score_intelligence,i_score_funny,i_score_ambition,i_score_shared_interests,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match,age_gap,age_gap_dir
0,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,35.000000,20.000000,20.000000,20.000000,0.000000,5.000000,6.0,8.0,8.0,8.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,6.0,9.0,7.0,7.0,6.0,5.0,0.14,3.0,2.0,7.0,6.0,0,6.0,positive
1,female,21.0,22.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,60.000000,0.000000,0.000000,40.000000,0.000000,0.000000,7.0,8.0,10.0,7.0,7.0,5.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,8.0,7.0,8.0,5.0,6.0,0.54,3.0,2.0,7.0,5.0,0,1.0,positive
2,female,21.0,22.0,Asian/PacificIslander/Asian-American,Asian/PacificIslander/Asian-American,2.0,4.0,19.000000,18.000000,19.000000,18.000000,14.000000,12.000000,10.0,10.0,10.0,10.0,10.0,10.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,8.0,9.0,8.0,5.0,7.0,0.16,3.0,2.0,7.0,-99.0,1,1.0,positive
3,female,21.0,23.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,30.000000,5.000000,15.000000,40.000000,5.000000,5.000000,7.0,8.0,9.0,8.0,9.0,8.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,6.0,8.0,7.0,6.0,8.0,0.61,3.0,2.0,7.0,6.0,1,2.0,positive
4,female,21.0,24.0,Asian/PacificIslander/Asian-American,Latino/HispanicAmerican,2.0,4.0,30.000000,10.000000,20.000000,10.000000,10.000000,20.000000,8.0,7.0,9.0,6.0,9.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,6.0,7.0,7.0,6.0,6.0,0.21,3.0,2.0,6.0,6.0,1,3.0,positive
5,female,21.0,25.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,50.000000,0.000000,30.000000,10.000000,0.000000,10.000000,7.0,7.0,8.0,8.0,7.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,4.0,9.0,7.0,4.0,6.0,4.0,0.25,3.0,2.0,6.0,5.0,0,4.0,positive
6,female,21.0,30.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,35.000000,15.000000,25.000000,10.000000,5.000000,10.000000,3.0,6.0,7.0,5.0,8.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,6.0,7.0,4.0,6.0,7.0,0.34,3.0,2.0,6.0,5.0,0,9.0,positive
7,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,33.333333,11.111111,11.111111,11.111111,11.111111,22.222222,6.0,7.0,5.0,6.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,4.0,9.0,7.0,6.0,5.0,6.0,0.50,3.0,2.0,6.0,7.0,0,6.0,positive
8,female,21.0,28.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,50.000000,0.000000,25.000000,10.000000,0.000000,15.000000,7.0,7.0,8.0,8.0,8.0,9.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,6.0,8.0,9.0,8.0,8.0,0.28,3.0,2.0,7.0,7.0,1,7.0,positive
9,female,21.0,24.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.0,6.0,6.0,6.0,6.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,6.0,6.0,8.0,10.0,8.0,-0.36,3.0,2.0,6.0,6.0,0,3.0,positive


## 3-7. 파생 변수: 인종 일치 점수 (Same Race Point)

단순히 같은 인종인지(0/1)로만 표현하면 정보가 부족하다.

**핵심 아이디어:** 인종의 **중요도(importance_same_race)**를 반영한 점수를 만든다.
- 같은 인종 → `+중요도` 점수
- 다른 인종 → `-중요도` 점수

예: 중요도가 2점인 사람이 같은 인종이면 +2, 다른 인종이면 -2

→ 데이터가 가지고 있는 정보를 더 잘 끌어내는 피처 엔지니어링

In [49]:
dating_df['importance_same_race'].value_counts()

importance_same_race
1.0     2749
3.0      964
2.0      938
5.0      644
8.0      631
7.0      536
6.0      516
4.0      494
9.0      404
10.0     246
0.0        8
Name: count, dtype: int64

In [50]:
dating_df['race'].unique()

array(['Asian/PacificIslander/Asian-American',
       'European/Caucasian-American', 'Other', 'Latino/HispanicAmerican',
       'Black/AfricanAmerican'], dtype=object)

In [51]:
dating_df['race_o'].unique()

array(['European/Caucasian-American',
       'Asian/PacificIslander/Asian-American', 'Latino/HispanicAmerican',
       'Other', 'Black/AfricanAmerican'], dtype=object)

In [52]:
# 같은 인종이면 1, 다르면 0
dating_df['same_race'] = (dating_df['race'] == dating_df['race_o']).astype('int')

In [53]:
dating_df.head()

,gender,age,age_o,race,race_o,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,i_score_attractive,i_score_sincere,i_score_intelligence,i_score_funny,i_score_ambition,i_score_shared_interests,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match,age_gap,age_gap_dir,same_race
0,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,35.0,20.0,20.0,20.0,0.0,5.0,6.0,8.0,8.0,8.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,6.0,9.0,7.0,7.0,6.0,5.0,0.14,3.0,2.0,7.0,6.0,0,6.0,positive,0
1,female,21.0,22.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,60.0,0.0,0.0,40.0,0.0,0.0,7.0,8.0,10.0,7.0,7.0,5.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,8.0,7.0,8.0,5.0,6.0,0.54,3.0,2.0,7.0,5.0,0,1.0,positive,0
2,female,21.0,22.0,Asian/PacificIslander/Asian-American,Asian/PacificIslander/Asian-American,2.0,4.0,19.0,18.0,19.0,18.0,14.0,12.0,10.0,10.0,10.0,10.0,10.0,10.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,8.0,9.0,8.0,5.0,7.0,0.16,3.0,2.0,7.0,-99.0,1,1.0,positive,1
3,female,21.0,23.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,30.0,5.0,15.0,40.0,5.0,5.0,7.0,8.0,9.0,8.0,9.0,8.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,6.0,8.0,7.0,6.0,8.0,0.61,3.0,2.0,7.0,6.0,1,2.0,positive,0
4,female,21.0,24.0,Asian/PacificIslander/Asian-American,Latino/HispanicAmerican,2.0,4.0,30.0,10.0,20.0,10.0,10.0,20.0,8.0,7.0,9.0,6.0,9.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,6.0,7.0,7.0,6.0,6.0,0.21,3.0,2.0,6.0,6.0,1,3.0,positive,0


In [54]:
# 0(다른 인종)을 -1로 바꿔서 방향성 부여
dating_df['same_race'] = dating_df['same_race'].replace({0: -1})

In [55]:
# 인종 일치 × 중요도 = 인종 일치 점수
dating_df['same_race_point'] = dating_df['same_race'] * dating_df['importance_same_race']

In [56]:
dating_df['same_race_point'].head(30)

0    -2.0
1    -2.0
2     2.0
3    -2.0
4    -2.0
5    -2.0
6    -2.0
7    -2.0
8    -2.0
9    -2.0
10    2.0
11    2.0
12   -2.0
13    2.0
14   -2.0
15    2.0
16    2.0
17    2.0
18    2.0
19    2.0
20    8.0
21    8.0
22   -8.0
23    8.0
24   -8.0
25    8.0
26    8.0
27    8.0
28    8.0
29    8.0
Name: same_race_point, dtype: float64

## 3-8. Rating 계산 (중요도 × 점수)

각 항목에 대해 `중요도(important) × 평가점수(score)` = **Rating**을 계산한다.

**주의사항:**
- score가 `-99`(결측)이면 rating도 `-99`
- important가 `0` 또는 `-99`이면 rating도 `-99` (중요도 0은 '응답 안함'과 동일 취급)
- 나중에 평균 계산할 때 `-99`인 항목은 제외해야 하므로 미리 표시

**`zip()` 함수로 여러 리스트를 동시 순회:**
- `for i, j, k in zip(important, score, rating)` → 3개 리스트를 병렬로 반복

In [57]:
o_important = []
o_score = []
i_important = []
i_score = []

for i in dating_df.columns:
    if i.startswith('o_important'):
        o_important.append(i)
    elif i.startswith('o_score'):
        o_score.append(i)
    elif i.startswith('i_important'):
        i_important.append(i)
    elif i.startswith('i_score'):
        i_score.append(i)

In [58]:
# important가 0인 경우도 결측 처리 (-99)
dating_df[o_important] = dating_df[o_important].replace({0: -99})

In [59]:
dating_df[i_important] = dating_df[i_important].replace({0: -99})

In [60]:
dating_df.head()

,gender,age,age_o,race,race_o,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,i_score_attractive,i_score_sincere,i_score_intelligence,i_score_funny,i_score_ambition,i_score_shared_interests,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match,age_gap,age_gap_dir,same_race,same_race_point
0,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,35.0,20.0,20.0,20.0,-99.0,5.0,6.0,8.0,8.0,8.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,6.0,9.0,7.0,7.0,6.0,5.0,0.14,3.0,2.0,7.0,6.0,0,6.0,positive,-1,-2.0
1,female,21.0,22.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,60.0,-99.0,-99.0,40.0,-99.0,-99.0,7.0,8.0,10.0,7.0,7.0,5.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,8.0,7.0,8.0,5.0,6.0,0.54,3.0,2.0,7.0,5.0,0,1.0,positive,-1,-2.0
2,female,21.0,22.0,Asian/PacificIslander/Asian-American,Asian/PacificIslander/Asian-American,2.0,4.0,19.0,18.0,19.0,18.0,14.0,12.0,10.0,10.0,10.0,10.0,10.0,10.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,8.0,9.0,8.0,5.0,7.0,0.16,3.0,2.0,7.0,-99.0,1,1.0,positive,1,2.0
3,female,21.0,23.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,30.0,5.0,15.0,40.0,5.0,5.0,7.0,8.0,9.0,8.0,9.0,8.0,15.0,20.0,20.0,15.0,15.0,15.0,7.0,6.0,8.0,7.0,6.0,8.0,0.61,3.0,2.0,7.0,6.0,1,2.0,positive,-1,-2.0
4,female,21.0,24.0,Asian/PacificIslander/Asian-American,Latino/HispanicAmerican,2.0,4.0,30.0,10.0,20.0,10.0,10.0,20.0,8.0,7.0,9.0,6.0,9.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,5.0,6.0,7.0,7.0,6.0,6.0,0.21,3.0,2.0,6.0,6.0,1,3.0,positive,-1,-2.0


In [61]:
# 단일 항목 테스트
dating_df['o_important_attractive'] * dating_df['o_score_attractive']

0       210.000000
1       420.000000
2       190.000000
3       210.000000
4       240.000000
           ...    
8372     80.000000
8373    105.263158
8374    300.000000
8376     50.000000
8377    160.000000
Length: 8130, dtype: float64

- 단순 곱셈으로 하면 `-99`인 값도 곱해져서 비정상 값이 나온다
- 함수를 정의해서 `-99`인 경우를 별도 처리해야 한다

In [62]:
def rating(data, important, score):
    if data[score] == -99:
        return -99
    elif data[important] == -99:
        return -99
    else:
        return data[important] * data[score]

In [63]:
# 단일 항목 테스트
dating_df.apply(lambda x: rating(x, 'o_important_attractive', 'o_score_attractive'), axis = 1)

0       210.000000
1       420.000000
2       190.000000
3       210.000000
4       240.000000
           ...    
8372     80.000000
8373    105.263158
8374    300.000000
8376     50.000000
8377    160.000000
Length: 8130, dtype: float64

### zip()으로 여러 리스트 동시 순회

- `zip(리스트A, 리스트B)` : 두 리스트를 병렬로 묶어서 반복
- important, score, rating 3개 리스트를 zip으로 한 번에 순회하면서 전체 rating을 계산

In [64]:
i_score

['i_score_attractive',
 'i_score_sincere',
 'i_score_intelligence',
 'i_score_funny',
 'i_score_ambition',
 'i_score_shared_interests']

In [65]:
sample_a = ['a','b','c']
sample_b = [1,2,3]

In [66]:
for i, j in zip(sample_a, sample_b):
    print(i, j)

a 1
b 2
c 3


In [67]:
for i, j in zip(i_important, i_score):
    print(i, j)

i_important_attractive i_score_attractive
i_important_sincere i_score_sincere
i_important_intellicence i_score_intelligence
i_important_funny i_score_funny
i_important_ambtition i_score_ambition
i_important_shared_interests i_score_shared_interests


In [68]:
o_rating = ['o_rating_attractive',
 'o_rating_sincere',
 'o_rating_intellicence',
 'o_rating_funny',
 'o_rating_ambtition',
 'o_rating_shared_interests']

In [69]:
i_rating = ['i_rating_attractive',
 'i_rating_sincere',
 'i_rating_intellicence',
 'i_rating_funny',
 'i_rating_ambtition',
 'i_rating_shared_interests']

In [70]:
# o_rating 전체 계산 (zip으로 3개 리스트 동시 순회)
for i, j, k in zip(o_important, o_score, o_rating):
     dating_df[k] = dating_df.apply(lambda x: rating(x, i, j), axis = 1)

In [71]:
# i_rating 전체 계산
for i, j, k in zip(i_important, i_score, i_rating):
     dating_df[k] = dating_df.apply(lambda x: rating(x, i, j), axis = 1)

### Rating 평균 계산 시 -99 제외

- 단순 `.mean()`을 쓰면 `-99` 값이 평균을 왜곡한다
- `x[x > 0].mean()` : 0보다 큰 값만 골라서 평균 계산
- 테스트할 때는 데이터 전체보다 **특정 행(예: -99가 많은 8377번)을 뽑아서** 확인하면 직관적

In [72]:
dating_df.head()

,gender,age,age_o,race,race_o,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,...,i_score_funny,i_score_ambition,i_score_shared_interests,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match,age_gap,age_gap_dir,same_race,same_race_point,o_rating_attractive,o_rating_sincere,o_rating_intellicence,o_rating_funny,o_rating_ambtition,o_rating_shared_interests,i_rating_attractive,i_rating_sincere,i_rating_intellicence,i_rating_funny,i_rating_ambtition,i_rating_shared_interests
0,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,35.0,20.0,20.0,20.0,-99.0,5.0,6.0,8.0,8.0,8.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,...,7.0,6.0,5.0,0.14,3.0,2.0,7.0,6.0,0,6.0,positive,-1,-2.0,210.0,160.0,160.0,160.0,-99.0,30.0,90.0,180.0,140.0,105.0,90.0,75.0
1,female,21.0,22.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,60.0,-99.0,-99.0,40.0,-99.0,-99.0,7.0,8.0,10.0,7.0,7.0,5.0,15.0,20.0,20.0,15.0,15.0,15.0,...,8.0,5.0,6.0,0.54,3.0,2.0,7.0,5.0,0,1.0,positive,-1,-2.0,420.0,-99.0,-99.0,280.0,-99.0,-99.0,105.0,160.0,140.0,120.0,75.0,90.0
2,female,21.0,22.0,Asian/PacificIslander/Asian-American,Asian/PacificIslander/Asian-American,2.0,4.0,19.0,18.0,19.0,18.0,14.0,12.0,10.0,10.0,10.0,10.0,10.0,10.0,15.0,20.0,20.0,15.0,15.0,15.0,...,8.0,5.0,7.0,0.16,3.0,2.0,7.0,-99.0,1,1.0,positive,1,2.0,190.0,180.0,190.0,180.0,140.0,120.0,75.0,160.0,180.0,120.0,75.0,105.0
3,female,21.0,23.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,30.0,5.0,15.0,40.0,5.0,5.0,7.0,8.0,9.0,8.0,9.0,8.0,15.0,20.0,20.0,15.0,15.0,15.0,...,7.0,6.0,8.0,0.61,3.0,2.0,7.0,6.0,1,2.0,positive,-1,-2.0,210.0,40.0,135.0,320.0,45.0,40.0,105.0,120.0,160.0,105.0,90.0,120.0
4,female,21.0,24.0,Asian/PacificIslander/Asian-American,Latino/HispanicAmerican,2.0,4.0,30.0,10.0,20.0,10.0,10.0,20.0,8.0,7.0,9.0,6.0,9.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,...,7.0,6.0,6.0,0.21,3.0,2.0,6.0,6.0,1,3.0,positive,-1,-2.0,240.0,70.0,180.0,60.0,90.0,140.0,75.0,120.0,140.0,105.0,90.0,90.0


In [73]:
# 8377번 행에서 -99가 아닌 값만 골라 평균 계산 (테스트)
dating_df[i_rating].loc[8377][dating_df[i_rating].loc[8377] > 0 ].mean()

np.float64(120.0)

In [74]:
# 전체 데이터에 적용: -99 제외하고 평균
dating_df['i_rating_total']= dating_df[i_rating].apply(lambda x: x[x > 0 ].mean(), axis = 1)

In [75]:
dating_df['o_rating_total']= dating_df[o_rating].apply(lambda x: x[x > 0 ].mean(), axis = 1)

In [76]:
dating_df.head()

,gender,age,age_o,race,race_o,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,...,i_score_shared_interests,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match,age_gap,age_gap_dir,same_race,same_race_point,o_rating_attractive,o_rating_sincere,o_rating_intellicence,o_rating_funny,o_rating_ambtition,o_rating_shared_interests,i_rating_attractive,i_rating_sincere,i_rating_intellicence,i_rating_funny,i_rating_ambtition,i_rating_shared_interests,i_rating_total,o_rating_total
0,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,35.0,20.0,20.0,20.0,-99.0,5.0,6.0,8.0,8.0,8.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,...,5.0,0.14,3.0,2.0,7.0,6.0,0,6.0,positive,-1,-2.0,210.0,160.0,160.0,160.0,-99.0,30.0,90.0,180.0,140.0,105.0,90.0,75.0,113.333333,144.000000
1,female,21.0,22.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,60.0,-99.0,-99.0,40.0,-99.0,-99.0,7.0,8.0,10.0,7.0,7.0,5.0,15.0,20.0,20.0,15.0,15.0,15.0,...,6.0,0.54,3.0,2.0,7.0,5.0,0,1.0,positive,-1,-2.0,420.0,-99.0,-99.0,280.0,-99.0,-99.0,105.0,160.0,140.0,120.0,75.0,90.0,115.000000,350.000000
2,female,21.0,22.0,Asian/PacificIslander/Asian-American,Asian/PacificIslander/Asian-American,2.0,4.0,19.0,18.0,19.0,18.0,14.0,12.0,10.0,10.0,10.0,10.0,10.0,10.0,15.0,20.0,20.0,15.0,15.0,15.0,...,7.0,0.16,3.0,2.0,7.0,-99.0,1,1.0,positive,1,2.0,190.0,180.0,190.0,180.0,140.0,120.0,75.0,160.0,180.0,120.0,75.0,105.0,119.166667,166.666667
3,female,21.0,23.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,30.0,5.0,15.0,40.0,5.0,5.0,7.0,8.0,9.0,8.0,9.0,8.0,15.0,20.0,20.0,15.0,15.0,15.0,...,8.0,0.61,3.0,2.0,7.0,6.0,1,2.0,positive,-1,-2.0,210.0,40.0,135.0,320.0,45.0,40.0,105.0,120.0,160.0,105.0,90.0,120.0,116.666667,131.666667
4,female,21.0,24.0,Asian/PacificIslander/Asian-American,Latino/HispanicAmerican,2.0,4.0,30.0,10.0,20.0,10.0,10.0,20.0,8.0,7.0,9.0,6.0,9.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,...,6.0,0.21,3.0,2.0,6.0,6.0,1,3.0,positive,-1,-2.0,240.0,70.0,180.0,60.0,90.0,140.0,75.0,120.0,140.0,105.0,90.0,90.0,103.333333,130.000000


In [77]:
dating_df.tail()

,gender,age,age_o,race,race_o,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,...,i_score_shared_interests,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match,age_gap,age_gap_dir,same_race,same_race_point,o_rating_attractive,o_rating_sincere,o_rating_intellicence,o_rating_funny,o_rating_ambtition,o_rating_shared_interests,i_rating_attractive,i_rating_sincere,i_rating_intellicence,i_rating_funny,i_rating_ambtition,i_rating_shared_interests,i_rating_total,o_rating_total
8372,male,25.0,24.0,European/Caucasian-American,European/Caucasian-American,1.0,1.0,10.000000,15.000000,30.000000,20.000000,15.000000,10.000000,8.0,8.0,7.0,7.0,8.0,6.0,70.0,-99.0,15.0,15.0,-99.0,-99.0,...,-99.0,0.28,10.0,-99.0,4.0,4.0,0,1.0,positive,1,1.0,80.000000,120.000000,210.000000,140.000000,120.000000,60.000000,490.0,-99.0,75.0,75.0,-99.0,-99.0,213.333333,121.666667
8373,male,25.0,26.0,European/Caucasian-American,Latino/HispanicAmerican,1.0,1.0,10.526316,10.526316,31.578947,21.052632,10.526316,15.789474,10.0,5.0,3.0,2.0,6.0,5.0,70.0,-99.0,15.0,15.0,-99.0,-99.0,...,-99.0,0.64,10.0,-99.0,2.0,5.0,0,1.0,negative,-1,-1.0,105.263158,52.631579,94.736842,42.105263,63.157895,78.947368,210.0,-99.0,75.0,75.0,-99.0,-99.0,120.000000,72.807018
8374,male,25.0,24.0,European/Caucasian-American,Other,1.0,1.0,50.000000,20.000000,10.000000,5.000000,10.000000,5.000000,6.0,3.0,7.0,3.0,7.0,2.0,70.0,-99.0,15.0,15.0,-99.0,-99.0,...,-99.0,0.71,10.0,-99.0,4.0,4.0,0,1.0,positive,-1,-1.0,300.000000,60.000000,70.000000,15.000000,70.000000,10.000000,280.0,-99.0,120.0,60.0,-99.0,-99.0,153.333333,87.500000
8376,male,25.0,22.0,European/Caucasian-American,Asian/PacificIslander/Asian-American,1.0,1.0,10.000000,25.000000,25.000000,10.000000,10.000000,20.000000,5.0,7.0,5.0,5.0,3.0,6.0,70.0,-99.0,15.0,15.0,-99.0,-99.0,...,5.0,0.62,10.0,-99.0,5.0,5.0,0,3.0,positive,-1,-1.0,50.000000,175.000000,125.000000,50.000000,30.000000,120.000000,280.0,-99.0,75.0,60.0,-99.0,-99.0,138.333333,91.666667
8377,male,25.0,22.0,European/Caucasian-American,Asian/PacificIslander/Asian-American,1.0,1.0,20.000000,20.000000,10.000000,15.000000,5.000000,30.000000,8.0,8.0,7.0,7.0,7.0,7.0,70.0,-99.0,15.0,15.0,-99.0,-99.0,...,1.0,0.01,10.0,-99.0,4.0,5.0,0,3.0,positive,-1,-1.0,160.000000,160.000000,70.000000,105.000000,35.000000,210.000000,210.0,-99.0,90.0,60.0,-99.0,-99.0,120.000000,123.333333


## 3-9. 조화 평균으로 최종 점수 산출

상대방 평가(o_rating_total)와 나의 평가(i_rating_total)를 합산할 때, 단순 산술 평균 대신 **조화 평균**을 사용한다.

**왜 조화 평균인가?**

- 산술 평균: `(a + b) / 2` → 한쪽이 극단적으로 높아도 평균이 올라감
- 조화 평균: `2ab / (a + b)` → 한쪽이 극단적으로 높아도 **낮은 쪽에 가깝게** 나옴

예:
- a=100, b=100 → 산술: 100, 조화: 100 (동일)
- a=50, b=150 → 산술: 100, 조화: **75** (더 보수적)

→ 양쪽 평가가 모두 높아야 최종 점수가 높아지는 구조

In [78]:
a = 100
b = 100

c = 50
d = 150

In [79]:
(a+b) /2

100.0

In [80]:
(c+d) / 2

100.0

In [81]:
2*a*b / (a+b)

100.0

In [82]:
2*c*d / (c+d)

75.0

In [83]:
# 조화 평균으로 최종 rating 계산
dating_df['rating_mean'] = 2 * dating_df['o_rating_total'] * dating_df['i_rating_total'] / (dating_df['o_rating_total'] + dating_df['i_rating_total'])

In [84]:
dating_df.head()

,gender,age,age_o,race,race_o,importance_same_race,importance_same_religion,o_important_attractive,o_important_sincere,o_important_intelligence,o_important_funny,o_important_ambitious,o_important_shared_interests,o_score_attractive,o_score_sincere,o_score_intelligence,o_score_funny,o_score_ambitous,o_score_shared_interests,i_important_attractive,i_important_sincere,i_important_intellicence,i_important_funny,i_important_ambtition,i_important_shared_interests,...,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,like,guess_prob_liked,match,age_gap,age_gap_dir,same_race,same_race_point,o_rating_attractive,o_rating_sincere,o_rating_intellicence,o_rating_funny,o_rating_ambtition,o_rating_shared_interests,i_rating_attractive,i_rating_sincere,i_rating_intellicence,i_rating_funny,i_rating_ambtition,i_rating_shared_interests,i_rating_total,o_rating_total,rating_mean
0,female,21.0,27.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,35.0,20.0,20.0,20.0,-99.0,5.0,6.0,8.0,8.0,8.0,8.0,6.0,15.0,20.0,20.0,15.0,15.0,15.0,...,0.14,3.0,2.0,7.0,6.0,0,6.0,positive,-1,-2.0,210.0,160.0,160.0,160.0,-99.0,30.0,90.0,180.0,140.0,105.0,90.0,75.0,113.333333,144.000000,126.839378
1,female,21.0,22.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,60.0,-99.0,-99.0,40.0,-99.0,-99.0,7.0,8.0,10.0,7.0,7.0,5.0,15.0,20.0,20.0,15.0,15.0,15.0,...,0.54,3.0,2.0,7.0,5.0,0,1.0,positive,-1,-2.0,420.0,-99.0,-99.0,280.0,-99.0,-99.0,105.0,160.0,140.0,120.0,75.0,90.0,115.000000,350.000000,173.118280
2,female,21.0,22.0,Asian/PacificIslander/Asian-American,Asian/PacificIslander/Asian-American,2.0,4.0,19.0,18.0,19.0,18.0,14.0,12.0,10.0,10.0,10.0,10.0,10.0,10.0,15.0,20.0,20.0,15.0,15.0,15.0,...,0.16,3.0,2.0,7.0,-99.0,1,1.0,positive,1,2.0,190.0,180.0,190.0,180.0,140.0,120.0,75.0,160.0,180.0,120.0,75.0,105.0,119.166667,166.666667,138.969874
3,female,21.0,23.0,Asian/PacificIslander/Asian-American,European/Caucasian-American,2.0,4.0,30.0,5.0,15.0,40.0,5.0,5.0,7.0,8.0,9.0,8.0,9.0,8.0,15.0,20.0,20.0,15.0,15.0,15.0,...,0.61,3.0,2.0,7.0,6.0,1,2.0,positive,-1,-2.0,210.0,40.0,135.0,320.0,45.0,40.0,105.0,120.0,160.0,105.0,90.0,120.0,116.666667,131.666667,123.713647
4,female,21.0,24.0,Asian/PacificIslander/Asian-American,Latino/HispanicAmerican,2.0,4.0,30.0,10.0,20.0,10.0,10.0,20.0,8.0,7.0,9.0,6.0,9.0,7.0,15.0,20.0,20.0,15.0,15.0,15.0,...,0.21,3.0,2.0,6.0,6.0,1,3.0,positive,-1,-2.0,240.0,70.0,180.0,60.0,90.0,140.0,75.0,120.0,140.0,105.0,90.0,90.0,103.333333,130.000000,115.142857


## 3-10. 최종 데이터 정리

- 분석에 필요한 컬럼만 선택
- 범주형 변수에 원-핫 인코딩 적용
- `pd.Series(df.columns)` : 컬럼 인덱스 번호를 확인하는 꿀팁

In [85]:
# 컬럼 이름을 직접 입력하지 않아도 되는 꿀팁: 인덱스 번호로 선택
select_col = list(dating_df.columns[[0,1,3]]) + list(dating_df.columns[31:])

In [86]:
select_col.remove('same_race')

In [87]:
select_col

['gender',
 'age',
 'race',
 'interests_correlate',
 'expected_happy_with_sd_people',
 'expected_num_interested_in_me',
 'like',
 'guess_prob_liked',
 'match',
 'age_gap',
 'age_gap_dir',
 'same_race_point',
 'o_rating_attractive',
 'o_rating_sincere',
 'o_rating_intellicence',
 'o_rating_funny',
 'o_rating_ambtition',
 'o_rating_shared_interests',
 'i_rating_attractive',
 'i_rating_sincere',
 'i_rating_intellicence',
 'i_rating_funny',
 'i_rating_ambtition',
 'i_rating_shared_interests',
 'i_rating_total',
 'o_rating_total',
 'rating_mean']

In [88]:
# 컬럼 인덱스 번호 확인용
pd.Series(dating_df.columns)

0                            gender
1                               age
2                             age_o
3                              race
4                            race_o
5              importance_same_race
6          importance_same_religion
7            o_important_attractive
8               o_important_sincere
9          o_important_intelligence
10                o_important_funny
11            o_important_ambitious
12     o_important_shared_interests
13               o_score_attractive
14                  o_score_sincere
15             o_score_intelligence
16                    o_score_funny
17                 o_score_ambitous
18         o_score_shared_interests
19           i_important_attractive
20              i_important_sincere
21         i_important_intellicence
22                i_important_funny
23            i_important_ambtition
24     i_important_shared_interests
25               i_score_attractive
26                  i_score_sincere
27             i_score_intel

In [89]:
final_df = dating_df[select_col]

In [90]:
# 범주형 변수 원-핫 인코딩
final_df = pd.get_dummies(final_df)